In [154]:
import pandas as pd
from pathlib import Path
from itertools import product, combinations

In [155]:
def read_data(TASKS = None, EXPERIMENTS = None, EMBS = None, MYTH_COMBOS = None):
    """Load all CSVs into a dict keyed by (task, experiment, emb, myth_combo)."""
    
    if TASKS is None: TASKS = ["2_AdviceGeneration", "3_Summarization"]
    if EXPERIMENTS is None: EXPERIMENTS = ["2_SemanticShift_Cosine", "3_MythShift_Projection"]
    if EMBS is None: EMBS = ["W2V", "GLOVE", "SBERT"]
    if MYTH_COMBOS is None: MYTH_COMBOS = ["singles", "pairs"]
        
    datasets = {}
    
    for task, experiment, emb, myth_combo in product(TASKS, EXPERIMENTS, EMBS, MYTH_COMBOS):
        data = pd.read_csv(f'{task}/SampleResults/{experiment}/{emb}_{myth_combo}.csv')
        datasets[(task, experiment, emb, myth_combo)] = data
    return datasets

In [156]:
# datasets = read_data(TASKS = ['3_Summarization'], EMBS = ['SBERT'])
datasets = read_data()

In [157]:
# datasets.keys()

In [57]:
MODELS = ["gemma", "llama", "mistral", "phi", "qwen"]
PROMPTS = ["p1", "p2", "p3"]

MYTH_TYPES = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]
MYTH_PAIRS = [f"{a}+{b}" for a, b in combinations(sorted(MYTH_TYPES), 2)]
MYTH_COL = {"singles": "myth_type", "pairs": "myth_pair"}

FRAMES = ["NegMyth", "NegNonMyth", "PosMyth", "PosNonMyth"]
DOSES  = [1, 2]

## T1 — Model × Myth Types

In [58]:
Path(f"2_AdviceGeneration/SampleResults/4_Tables").mkdir(exist_ok=True)
Path(f"3_Summarization/SampleResults/4_Tables").mkdir(exist_ok=True)

In [98]:
def chunk_tables(datasets):
    """ With meta data"""
    
    tables = {}
    for (task, exp, emb, combo), df in datasets.items():
        myth_col = MYTH_COL[combo]
        has_prompt = "prompt_variant" in df.columns and task == "2_AdviceGeneration"
        pv_vals = df["prompt_variant"].unique() if has_prompt else [None]

        for model in MODELS:
            m_df = df[df["model"] == model]
            for pv in pv_vals:
                pv_df = m_df[m_df["prompt_variant"] == pv] if pv else m_df
                base  = "_".join(filter(None, [task, exp, emb, combo, model, pv]))
                meta  = dict(task=task, exp=exp, emb=emb, combo=combo, model=model, pv=pv)

                tables[(base, 1)] = (pv_df, meta)
                for myth in pv_df[myth_col].dropna().unique():
                    myth_df = pv_df[pv_df[myth_col] == myth]
                    tables[(f"{base}_{myth}", 2)] = (myth_df, {**meta, "myth": myth})
                    for frame in FRAMES:
                        for dose in DOSES:
                            fd_df = myth_df[(myth_df["frame"]==frame)&(myth_df["dose"]==dose)]
                            if not fd_df.empty:
                                tables[(f"{base}_{myth}_{frame}_dose{dose}", 3)] = \
                                    (fd_df, {**meta, "myth": myth, "frame": frame, "dose": dose})
    return tables

In [129]:
tables = chunk_tables(datasets)

In [117]:
def significance_summary(tables, alpha=0.05, min_effect=0.0):
    rows = []
    for (k, l), (v, meta) in tables.items():
        rows.append({**meta, "level": l,
                     "significant": int((v["p_bh"] < alpha).any() and
                                        (v["cohens_dz"].abs() > min_effect).any())})
    df = pd.DataFrame(rows)
    return df.groupby(["emb","combo","model"])["significant"].agg(["sum","count"])

In [125]:
sig = {}
for task in ["2_AdviceGeneration", "3_Summarization"]:
    for exp in ["2_SemanticShift_Cosine", "3_MythShift_Projection"]:
        for emb in ["W2V", "GLOVE", "SBERT"]:
            sub_df = read_data(TASKS = [task], EXPERIMENTS = [exp], EMBS = [emb])
            sub_df_chunked = chunk_tables(sub_df)
            sig_sub_df_chunked = significance_summary(sub_df_chunked)
            sig[(task, exp, emb)] = sig_sub_df_chunked

In [127]:
# sig[('2_AdviceGeneration', '2_SemanticShift_Cosine', 'W2V')]

In [140]:
def diagnose(tables, alpha=0.05, out_dir="Tables"):
    """Diagnostic summary tables saved to Tables/."""
    Path(out_dir).mkdir(exist_ok=True)
    rows = []
    for (k, l), (v, meta) in tables.items():
        if l != 1: continue
        rows.append({
            **meta,
            "pct_sig":       (v["p_bh"] < alpha).mean() * 100,
            "mean_dz":       v["cohens_dz"].mean(),
            "max_dz":        v["cohens_dz"].abs().max(),
            "mean_mdiff":    v["mdiff"].mean(),
            "pct_pos_mdiff": (v["mdiff"] > 0).mean() * 100,
        })
    df = pd.DataFrame(rows)

    tables_out = {
        "1_direction_pos": df.groupby(["exp","model"])["pct_pos_mdiff"].mean().unstack("model").round(1),
        "2_mean_dz":        df.groupby(["model","exp"])["mean_dz"].mean().unstack("exp").round(2),
        "3_emb_agreement":  df.groupby(["emb","model","exp"])["mean_dz"].mean().unstack("exp").round(2),
        "4_cos_vs_proj":    _cos_proj_table(df),
    }

    for name, t in tables_out.items():
        t.to_csv(f"{out_dir}/{name}.txt")
    print(f"Saved {len(tables_out)} tables to {out_dir}/")

def _cos_proj_table(df):
    cos  = df[df["exp"].str.contains("Cosine")][["task","model","emb","combo","mean_dz"]].rename(columns={"mean_dz":"dz_cos"})
    proj = df[df["exp"].str.contains("Proj")][["task","model","emb","combo","mean_dz"]].rename(columns={"mean_dz":"dz_proj"})
    merged = cos.merge(proj, on=["task","model","emb","combo"])
    out = merged.groupby("model")[["dz_cos","dz_proj"]].mean().round(2)
    out.loc["correlation"] = [merged["dz_cos"].corr(merged["dz_proj"]), None]
    return out

In [141]:
tables = chunk_tables(datasets)
diagnose(tables)

Saved 4 tables to Tables/


In [152]:
def run_anova(datasets):
    """Type-II ANOVA on cohens_dz per (task, exp), with emb and combo as factors."""
    from statsmodels.formula.api import ols
    from statsmodels.stats.anova import anova_lm

    results = {}
    for task in ["2_AdviceGeneration", "3_Summarization"]:
        for exp in ["2_SemanticShift_Cosine", "3_MythShift_Projection"]:
            frames = []
            for (t, e, emb, combo), df in datasets.items():
                if t != task or e != exp: continue
                frames.append(df.assign(emb=emb, combo=combo))
            combined = pd.concat(frames, ignore_index=True).dropna(subset=["cohens_dz"])

            factors = ["C(model)", "C(myth_type)", "C(frame)", "C(dose)", "C(emb)", "C(combo)"]
            if "prompt_variant" in combined.columns and task == "2_AdviceGeneration":
                factors.append("C(prompt_variant)")

            formula = "cohens_dz ~ " + " + ".join(factors)
            try:
                fit = ols(formula, data=combined).fit()
                results[(task, exp)] = anova_lm(fit, typ=2)[["sum_sq", "df", "F", "PR(>F)"]]
            except Exception as ex:
                print(f"Failed: {task, exp} — {ex}")
    return results

In [158]:
run_anova(datasets)

{('2_AdviceGeneration',
  '2_SemanticShift_Cosine'):                         sum_sq      df           F         PR(>F)
 C(model)           1362.065990     4.0  605.632290  2.413299e-305
 C(myth_type)          6.770488     3.0    4.013928   7.399070e-03
 C(frame)             19.296043     3.0   11.439785   2.055542e-07
 C(dose)               0.201854     1.0    0.359011   5.491515e-01
 C(emb)              242.149816     2.0  215.340150   1.956781e-82
 C(combo)            506.066657     1.0  900.074770  1.147250e-153
 C(prompt_variant)    33.644454     2.0   29.919501   1.869491e-13
 Residual            800.643394  1424.0         NaN            NaN,
 ('2_AdviceGeneration',
  '3_MythShift_Projection'):                        sum_sq      df            F         PR(>F)
 C(model)           486.792354     4.0   679.188357   0.000000e+00
 C(myth_type)        27.022588     3.0    50.270379   6.995363e-31
 C(frame)             1.890363     3.0     3.516661   1.466097e-02
 C(dose)              0.